Импорты:

In [13]:
!pip install python-telegram-bot==20.7 nest-asyncio

ERROR:asyncio:Task was destroyed but it is pending!
task: <Task pending name='Application:8718579709:update_fetcher' coro=<Application._update_fetcher() running at /usr/local/lib/python3.12/dist-packages/telegram/ext/_application.py:1149> wait_for=<Future pending cb=[Task.__wakeup()]>>
ERROR:asyncio:Task was destroyed but it is pending!
task: <Task pending name='Application:8718579709:update_fetcher' coro=<Application._update_fetcher() running at /usr/local/lib/python3.12/dist-packages/telegram/ext/_application.py:1149> wait_for=<Future pending cb=[Task.__wakeup()]>>


In [14]:
import logging
import nest_asyncio
import asyncio
from telegram import Update, ReplyKeyboardMarkup, ReplyKeyboardRemove
from telegram.ext import Application, CommandHandler, MessageHandler, filters, ConversationHandler, ContextTypes
import os
from PIL import Image, ImageDraw, ImageFont
import textwrap

Дообучение модели:

In [15]:
# добавить

Добавление текста на фото:

In [16]:
def render_meme(image, caption):
    WIDTH = image.width
    PADDING = 20
    MAX_LINES = 3
    FONT_SIZE = 32

    font = ImageFont.truetype("DejaVuSans.ttf", FONT_SIZE)

    wrapped_lines = textwrap.wrap(caption, width=25)

    if len(wrapped_lines) > MAX_LINES:
        wrapped_lines = wrapped_lines[:MAX_LINES]

    dummy_img = Image.new("RGB", (WIDTH, 100))
    draw = ImageDraw.Draw(dummy_img)

    line_heights = []
    for line in wrapped_lines:
        bbox = draw.textbbox((0, 0), line, font=font)
        h = bbox[3] - bbox[1]
        line_heights.append(h)

    text_height = sum(line_heights) + PADDING * 2 + 10 * (len(wrapped_lines) - 1)

    new_height = image.height + text_height
    new_image = Image.new("RGB", (WIDTH, new_height), "white")

    new_image.paste(image, (0, 0))
    draw = ImageDraw.Draw(new_image)

    y_text = image.height + PADDING

    for line in wrapped_lines:
        bbox = draw.textbbox((0, 0), line, font=font)
        text_width = bbox[2] - bbox[0]

        x_text = (WIDTH - text_width) // 2

        draw.text((x_text, y_text), line, fill="black", font=font)

        y_text += (bbox[3] - bbox[1]) + 10

    return new_image

ТГ бот:

In [17]:
nest_asyncio.apply()
TOKEN = '8718579709:AAGjBJacPxzMcxIqMZFbLg8np3cgS2k0E0c'

In [18]:
logging.basicConfig(format='%(asctime)s - %(name)s - %(levelname)s - %(message)s', level=logging.INFO)

WAITING_FOR_DESCRIPTION = 1
WAITING_FOR_OVERLAY_TEXT = 2
user_memes = {}

async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Приветственное сообщение"""
    await update.message.reply_text(
        "🤖 *Привет! Я бот-мемогенератор*\n\n"
        "📋 *Доступные команды:*\n"
        "/creatememe - создать новый мем\n"
        "/help - список команд\n"
        "/cancel - отменить создание\n"
        "/status - статус генерации\n\n"
        "🎮 *Погнали!*",
        parse_mode="Markdown"
    )

async def help_command(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Помощь"""
    await update.message.reply_text(
        "📖 *Справочная информация*\n\n"
        "🔹 */creatememe* - запустить создание мема\n"
        "🔹 */cancel* - отменить текущее действие\n"
        "🔹 */status* - проверить статус бота\n"
        "🔹 */start* - приветствие\n\n"
        "*Как создать мем:*\n"
        "1️⃣ Введи /creatememe\n"
        "2️⃣ Опиши любой мем и укажи, какой текст на него добавить\n"
        "3️⃣ Бот покажет результат\n\n",
        parse_mode="Markdown"
    )

async def status(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Проверка статуса бота"""
    await update.message.reply_text(
        "✅ *Бот работает в штатном режиме*\n"
        "📊 uptime: активен",
        parse_mode="Markdown"
    )

async def cancel(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Отмена создания мема"""
    if update.effective_user.id in user_memes:
        del user_memes[update.effective_user.id]

    await update.message.reply_text(
        "❌ *Действие отменено*\n\n"
        "Чтобы создать новый мем - введи /creatememe",
        parse_mode="Markdown",
        reply_markup=ReplyKeyboardRemove()
    )
    return ConversationHandler.END

async def creatememe_start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Шаг 1: Запрос описания мема"""
    await update.message.reply_text(
        "🎨 *Шаг 1 из 2: Опиши мем*\n\n"
        "✏️ *Примеры описаний:*\n"
        "• «кот грустит из-за дождя»\n"
        "• «человек-паук опаздывает на работу»\n"
        "• «пельмень мечтает стать вареником»\n"
        "• «собака пытается понять физику»\n\n"
        "💬 *Напиши своё описание:*\n"
        "(Чтобы отменить - введи /cancel)",
        parse_mode="Markdown"
    )
    return WAITING_FOR_DESCRIPTION

async def process_meme_description(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Шаг 2: Запрос текста для мема"""
    user_id = update.effective_user.id
    description = update.message.text.strip()

    user_memes[user_id] = (description, None)

    await update.message.reply_text(
        f"✅ *Описание принято!*\n\n"
        "🎨 *Шаг 2 из 2: Какой текст добавить на мем?*\n\n"
        "✏️ *Примеры:*\n"
        "• «когда проснулся, а за окном зима»\n"
        "• «ничего не понимаю, но очень интересно»\n"
        "• «это печально»\n\n"
        "💬 *Напиши текст (или отправь «-» если без текста):*\n"
        "(Чтобы отменить - введи /cancel)",
        parse_mode="Markdown"
    )
    return WAITING_FOR_OVERLAY_TEXT


async def process_overlay_text(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Шаг 3: генерация мема"""
    user_id = update.effective_user.id
    overlay_text = update.message.text.strip()

    if user_id not in user_memes:
        await update.message.reply_text(
            "❌ *Что-то пошло не так*\n"
            "Начни заново с /creatememe",
            parse_mode="Markdown"
        )
        return ConversationHandler.END

    description, _ = user_memes[user_id]
    user_memes[user_id] = (description, overlay_text)

    if overlay_text == "-":
        overlay_text = None
        text_info = "без текста"
    else:
        text_info = f"«{overlay_text}»"

    thinking_msg = await update.message.reply_text(
        f"🎨 *Генерирую мем...*\n\n"
        f"⏳ Пожалуйста, подожди...",
        parse_mode="Markdown"
    )

   # try:
   #     prompt = description
    #    image_first = model(
   #         prompt=prompt,
   #         num_inference_steps=20,
   #         guidance_scale=7.5,
   #         height=512,
   #         width=512,
   #     ).images[0]
   #
   #     temp_image_path = f"temp_meme_{user_id}.png"
   #     image_first.save(temp_image_path)
  #      generated_image_path = temp_image_path

  # except Exception as e:
  #      print(f"[ERROR] Генерация не удалась: {e}")
  #      generated_image_path = None


# тест
    test_image_path = "Славам.png"
    generated_image_path = test_image_path if os.path.exists(test_image_path) else None
# тест

    if generated_image_path and overlay_text:
        try:
            img = Image.open(generated_image_path)
            img_with_text = render_meme(img, overlay_text)
            img_with_text.save(generated_image_path)
        except Exception as e:
            print(f"[ERROR] Наложение текста не удалось: {e}")

    final_image_path = generated_image_path

    await thinking_msg.delete()

    if final_image_path and os.path.exists(final_image_path):
        with open(final_image_path, 'rb') as photo:
            await update.message.reply_photo(
                photo=photo,
                caption=f"✅ *Мем сгенерирован!*\n\n"
                        f"📝 *Описание:* «{description}»\n"
                        f"💬 *Текст:* {text_info}\n\n"
                        f"✨ Хочешь ещё? Введи /creatememe",
                parse_mode="Markdown"
            )
        os.remove(final_image_path)
    else:
        await update.message.reply_text(
            f"❌ *Ошибка генерации*\n\n"
            f"Не удалось создать мем. Попробуй ещё раз.\n"
            f"Введи /creatememe",
            parse_mode="Markdown"
        )

    if user_id in user_memes:
        del user_memes[user_id]

    return ConversationHandler.END


async def handle_random_text(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Обработка случайного текста"""
    await update.message.reply_text(
        "❓ *Я тебя не понял*\n\n"
        "Вот что я умею:\n"
        "🎨 /creatememe - создать мем\n"
        "📖 /help - все команды\n"
        "❌ /cancel - отменить действие\n\n"
        "Просто напиши одну из команд 👆",
        parse_mode="Markdown"
    )

async def post_init(application: Application):
    """Действия после инициализации бота"""
    print("✅ Бот успешно запущен!")
    print("💡 Команды: /start, /creatememe, /help, /cancel, /status")

def main():
    """Запуск бота"""
    app = Application.builder().token(TOKEN).post_init(post_init).build()

    conv_handler = ConversationHandler(
        entry_points=[CommandHandler('creatememe', creatememe_start)],
        states={
            WAITING_FOR_DESCRIPTION: [
                MessageHandler(filters.TEXT & ~filters.COMMAND, process_meme_description)
            ],
            WAITING_FOR_OVERLAY_TEXT: [
                MessageHandler(filters.TEXT & ~filters.COMMAND, process_overlay_text)
            ],
        },
        fallbacks=[CommandHandler('cancel', cancel)],
        name="meme_creation",
        persistent=False
    )

    app.add_handler(conv_handler)
    app.add_handler(CommandHandler("start", start))
    app.add_handler(CommandHandler("help", help_command))
    app.add_handler(CommandHandler("status", status))
    app.add_handler(CommandHandler("cancel", cancel))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_random_text))

    try:
        loop = asyncio.get_running_loop()
        loop.create_task(app.run_polling())
    except RuntimeError:
        app.run_polling()

if __name__ == "__main__":
    main()

✅ Бот успешно запущен!
💡 Команды: /start, /creatememe, /help, /cancel, /status
✅ Бот успешно запущен!
💡 Команды: /start, /creatememe, /help, /cancel, /status


RuntimeError: Cannot close a running event loop